# 02 EDA Auxiliary Tables

This notebook focuses on the auxiliary Home Credit tables:

- `bureau`
- `bureau_balance`
- `previous_application`
- `POS_CASH_balance`
- `credit_card_balance`
- `installments_payments`


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(12)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from data_loading import read_table, scan_table

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR

PosixPath('/mnt/c/Users/arand/OneDrive/Desktop/NEU/ds4400/final_project/data/raw')

In [2]:
TABLES = {
    "bureau": "SK_ID_CURR",
    "bureau_balance": "SK_ID_BUREAU",
    "previous_application": "SK_ID_CURR",
    "pos_cash_balance": "SK_ID_CURR",
    "credit_card_balance": "SK_ID_CURR",
    "installments_payments": "SK_ID_CURR",
}

STATUS_COLUMNS = {
    "bureau": "CREDIT_ACTIVE",
    "bureau_balance": "STATUS",
    "previous_application": "NAME_CONTRACT_STATUS",
    "pos_cash_balance": "NAME_CONTRACT_STATUS",
    "credit_card_balance": "NAME_CONTRACT_STATUS",
}

def collect_streaming(lf):
    return lf.collect(engine="streaming")

def table_overview(table_name, id_col):
    sample = read_table(table_name, n_rows=5)
    row_count = collect_streaming(scan_table(table_name).select(pl.len().alias("n_rows")))[0, "n_rows"]
    return {
        "table": table_name,
        "id_col": id_col,
        "rows": int(row_count),
        "cols": len(sample.columns),
        "sample_columns": ", ".join(sample.columns[:8]),
    }

def top_missing(table_name, top_n=10):
    row_count = collect_streaming(scan_table(table_name).select(pl.len().alias("n_rows")))[0, "n_rows"]
    null_counts = collect_streaming(scan_table(table_name).select(pl.all().null_count()))
    missing = []
    for column in null_counts.columns:
        count = int(null_counts[0, column])
        missing.append({
            "column": column,
            "null_count": count,
            "null_pct": 100 * count / row_count,
        })
    return pd.DataFrame(missing).sort_values("null_pct", ascending=False).head(top_n)

def records_per_id_summary(table_name, id_col):
    counts = collect_streaming(
        scan_table(table_name)
        .group_by(id_col)
        .agg(pl.len().alias("record_count"))
    )
    s = counts["record_count"].to_pandas()
    return {
        "table": table_name,
        "id_col": id_col,
        "unique_ids": int(len(s)),
        "mean_records": float(s.mean()),
        "median_records": float(s.median()),
        "p95_records": float(s.quantile(0.95)),
        "max_records": int(s.max()),
    }, s

def status_counts(table_name, column):
    return collect_streaming(
        scan_table(table_name)
        .group_by(column)
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
    ).to_pandas()


In [ ]:
overview_df = pd.DataFrame([
    table_overview(table_name, id_col)
    for table_name, id_col in TABLES.items()
])
overview_df

In [ ]:
for table_name in TABLES:
    print(f"\n{table_name}")
    display(read_table(table_name, n_rows=5).to_pandas())

In [ ]:
record_summaries = []
record_distributions = {}

for table_name, id_col in TABLES.items():
    summary, distribution = records_per_id_summary(table_name, id_col)
    record_summaries.append(summary)
    record_distributions[table_name] = distribution

record_summary_df = pd.DataFrame(record_summaries)
record_summary_df.sort_values("mean_records", ascending=False)

In [ ]:
plot_df = record_summary_df.copy()
plot_df = plot_df.sort_values("mean_records", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=plot_df, x="mean_records", y="table", ax=axes[0], color="steelblue")
axes[0].set_title("Average Records Per ID")
axes[0].set_xlabel("Mean records")
axes[0].set_ylabel("")

sns.barplot(data=plot_df, x="p95_records", y="table", ax=axes[1], color="darkorange")
axes[1].set_title("95th Percentile of Records Per ID")
axes[1].set_xlabel("P95 records")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
for table_name, series in record_distributions.items():
    plt.figure(figsize=(8, 4))
    sns.histplot(series.clip(upper=series.quantile(0.99)), bins=50)
    plt.title(f"{table_name}: records per {TABLES[table_name]} (clipped at 99th pct)")
    plt.xlabel("record count")
    plt.show()

In [ ]:
for table_name in TABLES:
    print(f"\n{table_name}: top missing columns")
    display(top_missing(table_name, top_n=12))

In [ ]:
for table_name, column in STATUS_COLUMNS.items():
    counts = status_counts(table_name, column)
    print(f"\n{table_name}: {column}")
    display(counts)
    plt.figure(figsize=(8, 4))
    sns.barplot(data=counts, x="count", y=column, color="teal")
    plt.title(f"{table_name}: {column}")
    plt.show()